In [1]:
import pandas as pd
import ast
from bs4 import BeautifulSoup

def clean_and_describe_csv(file_path, title):
    # Lire le fichier CSV
    df = pd.read_csv(file_path)
    df = df.fillna(" ")  # Remplacer les NaN par une chaîne vide
    # Nettoyer les colonnes
    df['tags'] = df['tags'].apply(ast.literal_eval)
    df['stop_points'] = df['stop_points'].apply(ast.literal_eval)

    # Nettoyer la colonne 'text' si elle existe
    if 'text' in df.columns:
        df['text'] = df['text'].apply(lambda x: BeautifulSoup(x, "html.parser").get_text())
    """if 'total_valid_imp' in df.columns:
        if df['total_valid_imp'].values[0] == 0:
            df['total_valid_imp'].replace(0, 'valeur inconnu')"""

    descriptions = [title]
    for index, row in df.iterrows():
        description = (
            f"De {row['begin']} à {row['end']}, l'événement avec l'ID {row['id']} "
            f"(dernière mise à jour le {row['lastUpdate']}) est causé par {row['cause']}. "
            f"La sévérité est {row.get('severity_text', 'non spécifiée')}. "
            f"Les balises associées sont {', '.join(row['tags'])}. "
            f"Le titre de l'événement est : {row['title']}. "
            f"Message : {row['message']}. "
            f"Message court : {row['shortMessage']}. "
            f"Points d'arrêt affectés : {', '.join(row['stop_points'])}. "
            f"Nom : {row['name']}. Mode : {row['mode']}. "
        )
        if 'status' in row:
            description += f"Statut : {row['status']}. "
        if 'priority' in row:
            description += f"Priorité : {row['priority']}. "
        if 'effect' in row:
            description += f"Effet : {row['effect']}. "
        if 'text' in row:
            description += f"Texte : {row['text']}. "
        if 'total_valid_imp' in row:
            description += f"Voyageurs impactés : {'valeur inconnu' if row['total_valid_imp'] == 0 else row['total_valid_imp']}."        
        descriptions.append(description)
    return descriptions

# Traiter chaque CSV séparément avec des titres distinctifs
file_paths = ['df_current_final.csv', 'df_preview_final.csv']
titles = ["Perturbations en cours :", "Perturbations à venir :"]
descriptions_list = [clean_and_describe_csv(file_path, title) for file_path, title in zip(file_paths, titles)]

# Réunir les descriptions des deux CSV dans une seule variable
combined_descriptions = "\n\n".join(["\n".join(descriptions) for descriptions in descriptions_list])

# Afficher les descriptions combinées
print("Descriptions combinées des deux CSV :\n")
print(combined_descriptions)  # Limiter l'affichage à 1000 caractères pour éviter un trop long affichage

Descriptions combinées des deux CSV :

Perturbations en cours :
De 2025-03-27 18:42:00 à 2025-04-22 04:30:00, l'événement avec l'ID 26847bb8-0b33-11f0-b7af-0a58a9feac02 (dernière mise à jour le 20250327T184436) est causé par TRAVAUX. La sévérité est bloquante. Les balises associées sont Actualité. Le titre de l'événement est : Tramway T1 : Travaux - Trafic interrompu. Message : Jusqu'au lundi 21 avril 2025, le trafic est interrompu entre Gare de Noisy-le-Sec et Escadrille Normandie-Niémen en raison de travaux. .Bus de remplacement.Plus d'informations sur le site ratp.fr. Message court : Trafic interrompu. Points d'arrêt affectés : Auguste Delaune, Bobigny - Pablo Picasso, Escadrille Normandie-Niemen, Gare de Noisy-le-Sec, Hôtel de Ville de Bobigny, Jean Rostand, La Ferme, Libération, Petit Noisy, Pont de Bondy. Nom : T1. Mode : Tramway. Statut : now. Priorité : 0.0. Effet : NO_SERVICE. Texte : Jusqu'au lundi 21 avril 2025, le trafic est interrompu entre Gare de Noisy-le-Sec et Escadril

In [2]:
import requests
import json
import os # Recommended for handling API keys

# --- Configuration ---
# IMPORTANT: Replace with your actual key, preferably loaded securely.
# Example using an environment variable (recommended):
# API_KEY = os.getenv("OPENROUTER_API_KEY", "YOUR_FALLBACK_OR_TEST_KEY")
# For this specific example, using the key provided in the prompt:
API_KEY = "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"

myquestion="whats the given reference "

# Optional: Replace with your actual site info if desired
YOUR_SITE_URL = "http://localhost:8000" # Example placeholder
YOUR_SITE_NAME = "My Local Test"       # Example placeholder

API_URL = "https://openrouter.ai/api/v1/chat/completions"

# --- Prepare Request ---
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json", # Standard for JSON payloads
    "HTTP-Referer": YOUR_SITE_URL,      # Optional
    "X-Title": YOUR_SITE_NAME,          # Optional
}

data = {
    #"model": "openai/gpt-4o", # Specifies the model to use
     "model": "google/gemma-3-12b-it:free",
     "messages": [
        {
            "role": "user",
            "content": myquestion# Your question
        }
    ]
}

# --- Make API Call and Process Response ---
try:
    print("Sending request to OpenRouter...")
    response = requests.post(
        url=API_URL,
        headers=headers,
        data=json.dumps(data), # Convert Python dict to JSON string
        timeout=60 # Add a timeout (in seconds)
    )

    # Check if the request was successful (status code 2xx)
    response.raise_for_status()
    print("Request successful!")

    # Parse the JSON response
    response_data = response.json()
    # print("\n--- Full Raw Response ---")
    # print(json.dumps(response_data, indent=2)) # Uncomment to see the full structure
    # print("--- End Raw Response ---\n")


    # Extract the message content from the first choice
    if response_data.get("choices") and len(response_data["choices"]) > 0:
        message = response_data["choices"][0].get("message", {})
        ai_content = message.get("content")

        if ai_content:
            print("\n--- AI Answer ---")
            print(ai_content.strip()) # .strip() removes leading/trailing whitespace
            print("--- End AI Answer ---\n")
        else:
            print("Error: Could not find 'content' in the response message.")
            print("Response Message:", message)
    else:
        print("Error: 'choices' field not found or empty in the response.")
        print("Full Response Data:", response_data)

except requests.exceptions.HTTPError as http_err:
    print(f"HTTP error occurred: {http_err}")
    print(f"Response Status Code: {response.status_code}")
    try:
        # Try to print the error details from the API response body
        print(f"Response Body: {response.text}")
    except Exception:
        print("Could not read response body.")
except requests.exceptions.ConnectionError as conn_err:
    print(f"Connection error occurred: {conn_err}")
except requests.exceptions.Timeout as timeout_err:
    print(f"Request timed out: {timeout_err}")
except requests.exceptions.RequestException as req_err:
    print(f"An unexpected error occurred during the request: {req_err}")
except json.JSONDecodeError:
    print("Error: Could not decode the response JSON.")
    print("Raw Response Text:", response.text)
except KeyError as key_err:
    print(f"Error: Missing expected key in response JSON: {key_err}")
    print("Full Response Data:", response_data) # Show the data that caused the error
except IndexError:
    print("Error: Could not access expected index (e.g., choices[0]).")
    print("Full Response Data:", response_data)
except Exception as e:
    print(f"An unexpected general error occurred: {e}")

Sending request to OpenRouter...
Request successful!

--- AI Answer ---
Please provide me with the context! I need the text, document, or situation you're referring to in order to tell you what the "given reference" is. 😊 

For example, you could say:

* "In this article about climate change, what is the given reference?" (and then paste the article)
* "In the following paragraph, what is the given reference? [paragraph here]"
* "I'm looking at a historical document mentioning 'The Magna Carta.' What's the reference here?"

Once you give me the relevant information, I can help you identify the reference.
--- End AI Answer ---



In [2]:
import requests
import json
import os # Recommended for handling API keys

# --- Configuration ---
# IMPORTANT: Replace with your actual key, preferably loaded securely.
# Example using an environment variable (recommended):
# API_KEY = os.getenv("OPENROUTER_API_KEY", "YOUR_FALLBACK_OR_TEST_KEY")
# For this specific example, using the key provided in the prompt:
API_KEY = "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"

myquestion='''Rôle : Tu es un assistant qui rédige des synthèses humaines à partir de données structurées sur des perturbations ferroviaires. Pour chaque perturbation, rédige un **paragraphe clair, naturel et factuel**, en suivant les indications ci-dessous :
1. Commence par le nom de la ligne ou du mode de transport.
2. Indique la cause de la perturbation (par exemple, "en raison de travaux").
3. Précise la durée de l'interruption, avec les dates de début et de fin.
4. Mentionne si un service de remplacement est mis en place (par exemple, "bus de remplacement").
5. Cites les points d'arrêt affectés.
6. Si disponible, mentionne le nombre de voyageurs impactés.
7. Indique la sévérité de la perturbation (par exemple, "bloquante" ou "perturbée").
8. Indique le niveau de priorité et le statut de la perturbation.
9. Indique les perturbations en cours et à venir separement.
10. Precise le reprise de circulation prevu.(normalement l'heure end).
11. tu écrit seulement en français.

(exemple de réponse générée attendue pour une perturbation aléatoire : Sur le tramway T1, en raison de travaux, le trafic est totalement interrompu entre Gare de Noisy-le-Sec et Escadrille Normandie-Niémen jusqu'au lundi 21 avril 2025. Un bus de remplacement est mis en place pour desservir les points d'arrêt affectés, tels que Auguste Delaune, Bobigny - Pablo Picasso, et Gare de Noisy-le-Sec. Le nombre exact de voyageurs impactés est inconnu, mais cette perturbation est qualifiée de "bloquante".)

Données utilisées

Perturbations en cours :
Perturbation 1: Début: 2025-03-27 18:42:00 à Fin: 2025-04-22 04:30:00, Perturbation ID: 26847bb8-0b33-11f0-b7af-0a58a9feac02 (dernière mise à jour: 20250327T184436)
Cause: TRAVAUX. Sévérité: bloquante. Tag: Actualité. Title: Tramway T1: Travaux - Trafic interrompu. Message: Jusqu'au lundi 21 avril 2025, le trafic est interrompu entre Gare de Noisy-le-Sec et Escadrille Normandie-Niémen en raison de travaux. .Bus de remplacement.Plus d'informations sur le site ratp.fr. Message court: Trafic interrompu. Points d'arrêt affectés: Auguste Delaune, Bobigny - Pablo Picasso, Escadrille Normandie-Niemen, Gare de Noisy-le-Sec, Hôtel de Ville de Bobigny, Jean Rostand, La Ferme, Libération, Petit Noisy, Pont de Bondy. Nom: T1. Mode: Tramway. Statut perturbation: now. Niveau de priorité: 0. Effet: NO_SERVICE. Texte: Jusqu'au lundi 21 avril 2025, le trafic est interrompu entre Gare de Noisy-le-Sec et Escadrille Normandie-Niémen en raison de travaux. .Bus de remplacement.Plus d'informations sur le site ratp.fr. Voyageurs impactés: valeur inconnue.

Perturbation 2: Début: 2025-04-08 21:00:00 à Fin: 2025-04-09 02:59:00, Perturbation ID: 0dd19ac4-f819-11ef-af9a-0a58a9feac02 (dernière mise à jour: 20250303T111956)
Cause: TRAVAUX. Sévérité: perturbée. Tag: Actualité. Title: Ligne N: Paris Montp. - Dreux du 07/04 au 02/05.. Message: Période: du lundi au vendredi, le soir à partir de 21h00 Dates: du 07 avril au  02 mai 2025Sauf le Lundi 21 Avril et le jeudi 01ér maiLe trafic est interrompu entre Paris Montparnasse et Dreux.Dernier train PADO à destination de Paris Montparnasse départ à 21h52 de la gare de Dreux.Dernier train DAPO à destination de Dreux départ à 20h58  de la gare de Paris Montparnasse.Un service de bus de remplacement est mis en place, avec desserte des gares intermédiaires.Les horaires du calculateur d'itinéraire tiennent compte des travaux.Motif: travaux de maintenance.. Message court: Trafic interrompu planifié. Points d'arrêt affectés: Chaville Rive Gauche, Clamart, Dreux, Fontenay-le-Fleury, Garancières - La Queue, Gare Montparnasse, Gare de Bellevue, Gare de Villiers Neauphle Pontchartrain, Houdan, Marchezais - Broué, Meudon, Montfort-l'Amaury - Méré, Orgerus - Béhoust, Plaisir - Grignon, Plaisir - Les Clayes, Saint-Cyr, Sèvres Rive Gauche, Tacoignières - Richebourg, Vanves - Malakoff, Versailles Chantiers, Villepreux - Les Clayes, Viroflay Rive Gauche. Nom: N. Mode: LocalTrain. Statut perturbation: new. Niveau de priorité: 30. Effet: SIGNIFICANT_DELAYS. Texte: Période: du lundi au vendredi, le soir à partir de 21h00 Dates: du 07 avril au  02 mai 2025Sauf le Lundi 21 Avril et le jeudi 01ér maiLe trafic est interrompu entre Paris Montparnasse et Dreux.Dernier train PADO à destination de Paris Montparnasse départ à 21h52 de la gare de Dreux.Dernier train DAPO à destination de Dreux départ à 20h58  de la gare de Paris Montparnasse.Un service de bus de remplacement est mis en place, avec desserte des gares intermédiaires.Les horaires du calculateur d'itinéraire tiennent compte des travaux.Motif: travaux de maintenance.. Voyageurs impactés: 1171.

Perturbation 3: Début: 2025-04-08 20:40:00 à Fin: 2025-04-08 21:20:00, Perturbation ID: 97c3dd3e-0591-11f0-949f-0a58a9feac02 (dernière mise à jour: 20250320T144532)
Cause: TRAVAUX. Sévérité: perturbée. Tag: Actualité. Title: Ligne E: Villiers - Nanterre 7/04 - 9/05. Message: Dates: du lundi 7 avril au vendredi 9 mai sauf jeudi 8 maiLe train NOVY, au départ de Villiers-sur-Marne à 20h49, a pour terminus Paris Est et devient POVE.Il est sans arrêt de Rosa Parks à Paris Est. Ses horaires sont avancés jusqu'à 5 minutes à partir de Nogent-le-Perreux jusqu'au terminus. Les trains NOVY suivant et précédent ont leur desserte inchangée de Villiers-sur-Marne à Nanterre-la-Folie. Les horaires du calculateur d'itinéraire tiennent compte des travaux.Motif: travaux.. Message court: trafic ralenti. Points d'arrêt affectés: Haussmann Saint-Lazare, La Défense, Les Boullereaux Champigny, Magenta, Nanterre-La-Folie, Neuilly Porte Maillot, Nogent - Le Perreux, Noisy-le-Sec, Pantin, Rosa Parks, Rosny Bois Perrier, Rosny-sous-Bois, Val de Fontenay, Villiers-sur-Marne - Le Plessis-Trévise. Nom: E. Mode: RapidTransit. Statut perturbation: now. Niveau de priorité: 30. Effet: SIGNIFICANT_DELAYS. Texte: Dates: du lundi 7 avril au vendredi 9 mai sauf jeudi 8 maiLe train NOVY, au départ de Villiers-sur-Marne à 20h49, a pour terminus Paris Est et devient POVE.Il est sans arrêt de Rosa Parks à Paris Est. Ses horaires sont avancés jusqu'à 5 minutes à partir de Nogent-le-Perreux jusqu'au terminus. Les trains NOVY suivant et précédent ont leur desserte inchangée de Villiers-sur-Marne à Nanterre-la-Folie. Les horaires du calculateur d'itinéraire tiennent compte des travaux.Motif: travaux.. Voyageurs impactés: 826.


Perturbations à venir :
Perturbation 1: Début: 2025-04-08 22:00:00 à Fin: 2025-04-09 04:30:00, Perturbation ID: e0812420-ad56-11ef-b233-0a58a9feac02 (dernière mise à jour: 20241128T080331)
Cause: TRAVAUX. Sévérité: non spécifiée. Tag: Actualité. Title: Métro 14: Travaux - Trafic interrompu. Message: Du 3 mars au 15 avril, les lundi et mardi à partir de 22:00, le trafic sera interrompu entre Maison Blanche et Saint-Denis - Pleyel en raison de travaux. Bus entre Gare de Lyon et OlympiadesPlus d'informations sur le site ratp.fr. Message court:  . Points d'arrêt affectés: Bercy, Bibliothèque François Mitterrand, Châtelet, Cour Saint-Emilion, Gare de Lyon, Madeleine, Mairie de Saint-Ouen, Maison Blanche, Olympiades, Pont Cardinet, Porte de Clichy, Pyramides, Saint-Denis - Pleyel, Saint-Lazare, Saint-Ouen. Nom: 14. Mode: Metro.

Perturbation 2: Début: 2025-04-08 22:00:00 à Fin: 2025-04-09 04:30:00, Perturbation ID: 4e42f940-0f34-11f0-8f48-0a58a9feac02 (dernière mise à jour: 20250401T220258)
Cause: TRAVAUX. Sévérité: non spécifiée. Tag: Actualité. Title: Métro 14: Travaux - Trafic interrompu. Message: Jusqu'au 29 avril, les lundi et mardi à partir de 22:00, le trafic est interrompu entre Maison Blanche et Saint-Denis - Pleyel en raison de travaux. Bus entre Gare de Lyon et OlympiadesPlus d'informations sur le site ratp.fr. Message court: Trafic interrompu. Points d'arrêt affectés: Bercy, Bibliothèque François Mitterrand, Châtelet, Cour Saint-Emilion, Gare de Lyon, Madeleine, Mairie de Saint-Ouen, Maison Blanche, Olympiades, Pont Cardinet, Porte de Clichy, Pyramides, Saint-Denis - Pleyel, Saint-Lazare, Saint-Ouen. Nom: 14. Mode: Metro.

Perturbation 3: Début: 2025-04-08 21:55:00 à Fin: 2025-04-09 02:59:00, Perturbation ID: 212f0f32-f816-11ef-b418-0a58a9feac02 (dernière mise à jour: 20250303T105900)
Cause: TRAVAUX. Sévérité: non spécifiée. Tag: Actualité. Title: Ligne N: Paris M - Plaisir - G du 22/04 au 02/05. Message: Période: du lundi au vendredi, le soir à partir de 22h00Dates: du 07 avril au  02 mai sauf le 01er MaiLe trafic est interrompu entre Paris Montparnasse et Plaisir - Grignon.Dernier train GOPI à destination de Plaisir Grignon 23h05 au départ de Paris Montparnasse.Dernier train POGI à destination de Paris Montparnasse 21h56 au départ de Plaisir Grignon.Un service de bus de remplacement est mis en place, avec desserte des gares intermédiaires.Les horaires du calculateur d'itinéraire tiennent compte des travaux.Motif: travaux de maintenance.. Message court: Trafic interrompu planifié. Points d'arrêt affectés: Chaville Rive Gauche, Clamart, Fontenay-le-Fleury, Gare Montparnasse, Gare de Bellevue, Meudon, Plaisir - Grignon, Plaisir - Les Clayes, Saint-Cyr, Sèvres Rive Gauche, Vanves - Malakoff, Versailles Chantiers, Villepreux - Les Clayes, Viroflay Rive Gauche. Nom: N. Mode: LocalTrain.
'''

# Optional: Replace with your actual site info if desired
YOUR_SITE_URL = "http://localhost:8000" # Example placeholder
YOUR_SITE_NAME = "My Local Test"       # Example placeholder

API_URL = "https://openrouter.ai/api/v1/chat/completions"

# --- Prepare Request ---
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json", # Standard for JSON payloads
    "HTTP-Referer": YOUR_SITE_URL,      # Optional
    "X-Title": YOUR_SITE_NAME,          # Optional
}

data = {
    #"model": "openai/gpt-4o", # Specifies the model to use
     "model": "google/gemma-3-12b-it:free",
     "messages": [
        {
            "role": "user",
            "content": myquestion# Your question
        }
    ]
}

# --- Make API Call and Process Response ---
try:
    print("Sending request to OpenRouter...")
    response = requests.post(
        url=API_URL,
        headers=headers,
        data=json.dumps(data), # Convert Python dict to JSON string
        timeout=60 # Add a timeout (in seconds)
    )

    # Check if the request was successful (status code 2xx)
    response.raise_for_status()
    print("Request successful!")

    # Parse the JSON response
    response_data = response.json()
    # print("\n--- Full Raw Response ---")
    # print(json.dumps(response_data, indent=2)) # Uncomment to see the full structure
    # print("--- End Raw Response ---\n")


    # Extract the message content from the first choice
    if response_data.get("choices") and len(response_data["choices"]) > 0:
        message = response_data["choices"][0].get("message", {})
        ai_content = message.get("content")

        if ai_content:
            print("\n--- AI Answer ---")
            print(ai_content.strip()) # .strip() removes leading/trailing whitespace
            print("--- End AI Answer ---\n")
        else:
            print("Error: Could not find 'content' in the response message.")
            print("Response Message:", message)
    else:
        print("Error: 'choices' field not found or empty in the response.")
        print("Full Response Data:", response_data)

except requests.exceptions.HTTPError as http_err:
    print(f"HTTP error occurred: {http_err}")
    print(f"Response Status Code: {response.status_code}")
    try:
        # Try to print the error details from the API response body
        print(f"Response Body: {response.text}")
    except Exception:
        print("Could not read response body.")
except requests.exceptions.ConnectionError as conn_err:
    print(f"Connection error occurred: {conn_err}")
except requests.exceptions.Timeout as timeout_err:
    print(f"Request timed out: {timeout_err}")
except requests.exceptions.RequestException as req_err:
    print(f"An unexpected error occurred during the request: {req_err}")
except json.JSONDecodeError:
    print("Error: Could not decode the response JSON.")
    print("Raw Response Text:", response.text)
except KeyError as key_err:
    print(f"Error: Missing expected key in response JSON: {key_err}")
    print("Full Response Data:", response_data) # Show the data that caused the error
except IndexError:
    print("Error: Could not access expected index (e.g., choices[0]).")
    print("Full Response Data:", response_data)
except Exception as e:
    print(f"An unexpected general error occurred: {e}")

Sending request to OpenRouter...
Request successful!

--- AI Answer ---
]

Voici les synthèses des perturbations ferroviaires telles que demandées :

**Perturbations en cours :**

*   **Tramway T1 :** En raison de travaux, le trafic est totalement interrompu entre Gare de Noisy-le-Sec et Escadrille Normandie-Niémen jusqu'au lundi 21 avril 2025. Un bus de remplacement est mis en place pour desservir les points d'arrêt Auguste Delaune, Bobigny - Pablo Picasso, Escadrille Normandie-Niemen, Gare de Noisy-le-Sec, Hôtel de Ville de Bobigny, Jean Rostand, La Ferme, Libération, Petit Noisy, Pont de Bondy. Cette perturbation, qualifiée de "bloquante", n'a pas d'impact sur un nombre connu de voyageurs et est confirmée actuellement. La reprise de circulation est prévue pour le lundi 21 avril à 04h30.
*   **Ligne N :** Le trafic est interrompu entre Paris Montparnasse et Dreux du lundi au vendredi, de 21h00 à 02h59, jusqu'au 2 mai 2025, sauf le lundi 21 avril et le jeudi 1er mai, en raison de trav